# Multiple environments in one MouseEnv

Pass a `list[EnvConfig]` to `make_env` to combine several different environment types in one `MouseEnv`. Each config becomes an independent inner env with its own parallel slots. The envs are stepped sequentially and their outputs are never concatenated — you get one `(outputs, metrics)` tuple per inner env.

This notebook runs **CartPole-v1** (2 parallel slots) and **MountainCar-v0** (3 parallel slots) together, prints their specs side-by-side, and runs a short rollout loop.

## Imports

In [1]:
from mouse_envs import EnvConfig, make_env

## Build a multi-environment

`make_env` accepts a plain list of configs. Each entry is independent — different ids, different `num_envs`, different action/observation spaces are all fine.

In [2]:
env = make_env([
    EnvConfig(id="CartPole-v1",   seed=0, num_envs=2, max_episode_steps=500),
    EnvConfig(id="MountainCar-v0", seed=1, num_envs=3, max_episode_steps=200),
])

## Inspect specs

`env.output_specs` and `env.input_specs` are lists — one entry per inner env. `env.names` is the full flattened list of slot names across all inner envs.

In [3]:
print(f"Total parallel slots : {env.num_envs}")
print(f"All names            : {env.names}")
print()
for i, (ospec, ispec) in enumerate(zip(env.output_specs, env.input_specs)):
    print(f"Inner env {i}")
    print(f"  obs  dtype={ospec.observation.dtype}  shape={ospec.observation.shape}")
    print(f"  act  dtype={ispec.action.dtype}        shape={ispec.action.shape}")

Total parallel slots : 5
All names            : ('CartPole-v1#0', 'CartPole-v1#1', 'MountainCar-v0#0', 'MountainCar-v0#1', 'MountainCar-v0#2')

Inner env 0
  obs  dtype=torch.float32  shape=(4,)
  act  dtype=torch.int64        shape=()
Inner env 1
  obs  dtype=torch.float32  shape=(2,)
  act  dtype=torch.int64        shape=()


## Rollout loop

`env.sample_random_inputs()` returns `list[list[dict]]` — one `list[dict]` per inner env.
`env.step(inputs_per_env)` returns `list[tuple[outputs, metrics]]` — one tuple per inner env.

Iterate over `results` to process each inner env independently.

In [4]:
episode_rewards = [[] for _ in env.output_specs]  # one list per inner env

for step in range(300):
    results = env.step(env.sample_random_inputs())

    for env_idx, (outputs, metrics) in enumerate(results):
        for slot_metrics in metrics:
            episode_rewards[env_idx].extend(slot_metrics["episode_cum_reward"])

for env_idx, rewards in enumerate(episode_rewards):
    name = env.names[sum(e.num_envs for e in env._inners[:env_idx])]
    avg = sum(rewards) / len(rewards) if rewards else float("nan")
    print(f"{name.split('#')[0]:20s}  episodes={len(rewards):3d}  avg_return={avg:.2f}")

env.close()

CartPole-v1           episodes= 22  avg_return=24.82
MountainCar-v0        episodes=  3  avg_return=-200.00
